# contacts-v1 rollouts — what makes a high-accuracy rollout? (MarinFold #102)

Interactive analysis for [Open-Athena/MarinFold#102](https://github.com/Open-Athena/MarinFold/issues/102):
**what differentiates high-accuracy rollouts from average ones** across contact-prediction targets?

This notebook runs entirely on **CPU** (no GPU, no model, no auth) from the public **exp102** data:
~200 length-stratified contacts-v1 train targets × 1000 rollouts from the tuned 1.5B model, regenerated to
keep what exp98 discarded — each rollout's contacts in **emission order** plus each contact's **logprob**.

It's a scaffold with a plot per question, not a set of conclusions — fork it and dig in.

> Runs on a free Colab CPU runtime. The full per-contact table is ~18M rows; by default we use a
> 40-target subset so cells stay snappy — set `USE_ALL_TARGETS = True` in the setup cell for the full run.

## 0. Setup — load the published data (anonymous)

In [ ]:
# Colab already has pandas/numpy/scipy/matplotlib/pyarrow. Uncomment if a fresh env is missing one:
# !pip -q install pandas pyarrow scipy matplotlib
import io, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BUCKET = ('https://huggingface.co/buckets/open-athena/MarinFold/resolve/'
          'data/contacts-v1-rollouts-ordered-exp102')

def load_parquet(name):
    """Download a published parquet to memory and read it (anon, no HF login)."""
    data = urllib.request.urlopen(f'{BUCKET}/{name}', timeout=120).read()
    return pd.read_parquet(io.BytesIO(data))

m = load_parquet('rollout_metrics_ordered.parquet')  # one row per rollout
t = load_parquet('targets.parquet')                  # one row per target (sequence + gt_contacts)
print(f'{len(m):,} rollouts across {m.entry_id.nunique()} targets; {len(t)} targets loaded')
m.head(3)

**Schema** — `m` (`rollout_metrics_ordered.parquet`), one row per rollout, keyed `entry_id`+`r`:

| column | meaning |
|---|---|
| `pred` | flattened predicted contacts `[i0,j0,i1,j1,…]` **in emission order** (deduped, seq-index, sep≥6). Index = prediction rank. |
| `pred_logprob` | length `n_pred`; contact *k*'s emission logprob (higher = more confident). |
| `all_f1`,`all_prec`,`all_rec` … | per-band (`all`/`short`/`med`/`long`) precision/recall/F1. |
| `n_gen_tokens`,`nll`,`nll_per_tok`,`finished`,`n_pred` | rollout length, total NLL, per-token NLL, hit `<end>`, #distinct contacts. |

`t` (`targets.parquet`): `entry_id`, `L`, `sequence` (one-letter), `gt_contacts` (ground-truth `[i,j]`).
Separation, residue identities, and correctness are all derived below.

In [ ]:
USE_ALL_TARGETS = False   # True = full 18M-contact table (needs ~a few GB RAM)
N_DEMO = 40               # targets to use when USE_ALL_TARGETS is False

if not USE_ALL_TARGETS:
    keep = sorted(m.entry_id.unique())[:N_DEMO]
    m = m[m.entry_id.isin(keep)].copy()
    t = t[t.entry_id.isin(keep)].copy()
    print(f'demo subset: {len(m):,} rollouts across {m.entry_id.nunique()} targets')

### Build the two workhorse tables

`m` also gets a per-target F1 **quartile** (`rollout_q`), so "high-accuracy" is defined relative to each
target's own rollout distribution (Q4 = the good rollouts). `contacts_frame` explodes every rollout into
one row per predicted contact with its emission `rank`, separation `sep`/`band`, `logprob`, whether it's
`correct`, and the endpoint residues `aa_i`/`aa_j`.

In [ ]:
m['rollout_q'] = (m.groupby('entry_id')['all_f1']
                  .transform(lambda s: pd.qcut(s.rank(method='first'), 4,
                                               labels=['Q1','Q2','Q3','Q4'])))

def band_of(sep):
    return 'short' if sep <= 11 else ('med' if sep <= 23 else 'long')

def contacts_frame(m, t):
    gt  = {r.entry_id: {(int(i), int(j)) for i, j in r.gt_contacts} for r in t.itertuples()}
    seq = {r.entry_id: r.sequence for r in t.itertuples()}
    Lm  = {r.entry_id: int(r.L) for r in t.itertuples()}
    rows = []
    for e in m.itertuples():
        g, s = gt[e.entry_id], seq[e.entry_id]
        for rank in range(len(e.pred_logprob)):
            i, j = int(e.pred[2*rank]), int(e.pred[2*rank+1])
            sep = abs(i - j)
            rows.append((e.entry_id, e.r, rank, i, j, sep, band_of(sep),
                         float(e.pred_logprob[rank]), (i, j) in g, float(e.all_f1),
                         e.rollout_q, Lm[e.entry_id],
                         s[i] if i < len(s) else '?', s[j] if j < len(s) else '?'))
    return pd.DataFrame(rows, columns=['entry_id','r','rank','i','j','sep','band',
                                       'logprob','correct','rollout_f1','rollout_q','L','aa_i','aa_j'])

cf = contacts_frame(m, t)
cf['norm_rank'] = cf.groupby(['entry_id','r'])['rank'].transform(lambda s: s/max(len(s)-1,1))
print(f'contacts_frame: {len(cf):,} (rollout, contact) rows')
cf.head(3)

## 1. Are high-accuracy rollouts just longer?

Per target, correlate each rollout's F1 with its length (generated tokens) and #contacts, then look at the
distribution of those within-target Spearman correlations across targets. A tight bump near 0 ⇒ length isn't
the driver; a shift to the right ⇒ longer rollouts tend to score higher.

In [ ]:
def per_target_len_corr(m):
    def sp(g):
        return pd.Series({
            'rho_f1_vs_ntok':  g['all_f1'].corr(g['n_gen_tokens'], method='spearman'),
            'rho_f1_vs_npred': g['all_f1'].corr(g['n_pred'],       method='spearman')})
    return m.groupby('entry_id').apply(sp, include_groups=False)

lc = per_target_len_corr(m)
print('mean within-target Spearman:'); print(lc.mean().round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11,3.4))
ax[0].hist(lc['rho_f1_vs_ntok'].dropna(), bins=30, color='#4C72B0')
ax[0].axvline(0, color='k', lw=.8); ax[0].axvline(lc['rho_f1_vs_ntok'].mean(), color='#C44E52', ls='--', label='mean')
ax[0].set_title('within-target Spearman(F1, n_gen_tokens)'); ax[0].set_xlabel('rho'); ax[0].legend()
# F1 vs length, one representative target
ex = m.groupby('entry_id').size().index[0]
g = m[m.entry_id == ex]
ax[1].scatter(g['n_gen_tokens'], g['all_f1'], s=6, alpha=.25, color='#55A868')
ax[1].set_title(f'one target ({ex}): F1 vs rollout length'); ax[1].set_xlabel('n_gen_tokens'); ax[1].set_ylabel('F1')
plt.tight_layout(); plt.show()

## 2. Do high-accuracy rollouts emit contacts in a different order?

For each rollout, contacts have a normalized emission rank (0 = first, 1 = last). If good rollouts front-load
a separation band (e.g. **long-range first**), that band's mean rank should be lower in Q4 than Q1.
Below: mean normalized rank by band × F1-quartile (lower = earlier).

In [ ]:
ob = (cf.pivot_table(index='rollout_q', columns='band', values='norm_rank',
                     aggfunc='mean', observed=True)
        .loc[['Q1','Q2','Q3','Q4'], ['short','med','long']])
print(ob.round(3).to_string())

fig, ax = plt.subplots(figsize=(6.5,3.6))
for band in ['short','med','long']:
    ax.plot(['Q1','Q2','Q3','Q4'], ob[band], marker='o', label=band)
ax.set_ylabel('mean normalized emission rank\n(lower = emitted earlier)')
ax.set_xlabel('rollout F1 quartile (Q4 = best)'); ax.set_title('order bias by separation band')
ax.legend(title='band'); plt.tight_layout(); plt.show()

## 3. Confidence & position — are the good rollouts confident early, or just accurate throughout?

Two probes per rollout: (a) Spearman(rank, logprob) — negative ⇒ most-confident contacts emitted first;
(b) correctness of the **first 3** vs **last 3** emitted contacts. Averaged by F1 quartile. If Q4's edge is
concentrated early, `acc_first3` should outrun `acc_last3` more in Q4.

In [ ]:
def per_rollout_conf(g):
    g = g.sort_values('rank')
    rho = g['rank'].corr(g['logprob'], method='spearman') if len(g) > 3 else np.nan
    return pd.Series({'rho_rank_logprob': rho,
                      'acc_first3': g.head(3)['correct'].mean(),
                      'acc_last3':  g.tail(3)['correct'].mean()})

conf = (cf.groupby(['entry_id','r','rollout_q'], observed=True)
          .apply(per_rollout_conf, include_groups=False).reset_index()
          .groupby('rollout_q', observed=True)[['rho_rank_logprob','acc_first3','acc_last3']]
          .mean().loc[['Q1','Q2','Q3','Q4']])
print(conf.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11,3.6))
x = np.arange(4)
ax[0].bar(x-0.2, conf['acc_first3'], 0.4, label='first 3', color='#4C72B0')
ax[0].bar(x+0.2, conf['acc_last3'],  0.4, label='last 3',  color='#DD8452')
ax[0].set_xticks(x); ax[0].set_xticklabels(conf.index); ax[0].set_ylabel('correctness')
ax[0].set_title('early vs late contact accuracy by quartile'); ax[0].legend()
ax[1].plot(conf.index, conf['rho_rank_logprob'], marker='o', color='#C44E52')
ax[1].set_title('Spearman(emission rank, logprob)'); ax[1].set_xlabel('F1 quartile')
ax[1].axhline(0, color='k', lw=.8); plt.tight_layout(); plt.show()

## 4. Do high- vs low-accuracy rollouts favor contacts between different amino acids?

Log2 enrichment of each residue (either endpoint of a predicted contact) in Q4 vs Q1 rollouts.
Bars far from 0 ⇒ that residue is over/under-represented among the good rollouts' predicted contacts.

In [ ]:
def comp(q):
    s = pd.concat([cf.loc[cf.rollout_q==q,'aa_i'], cf.loc[cf.rollout_q==q,'aa_j']])
    return s.value_counts(normalize=True)
q4, q1 = comp('Q4'), comp('Q1')
aa = sorted(set(q4.index) | set(q1.index))
enr = pd.DataFrame({'Q4':q4.reindex(aa).fillna(0), 'Q1':q1.reindex(aa).fillna(0)})
enr['log2_enrich_Q4'] = np.log2((enr.Q4+1e-6)/(enr.Q1+1e-6))
enr = enr.sort_values('log2_enrich_Q4')

fig, ax = plt.subplots(figsize=(9,3.4))
bar_colors = ['#C44E52' if v < 0 else '#4C72B0' for v in enr['log2_enrich_Q4']]
ax.bar(enr.index, enr['log2_enrich_Q4'], color=bar_colors)
ax.axhline(0, color='k', lw=.8); ax.set_ylabel('log2(Q4 / Q1) residue frac')
ax.set_title('amino-acid enrichment in high-F1 (Q4) vs low-F1 (Q1) rollouts')
plt.tight_layout(); plt.show()

## 5. Within one target — is there a single informative contact the good rollouts guess early?

Pick the target with the most rollout-to-rollout F1 variance. For each ground-truth contact, measure how
strongly its **earliness** (1 − normalized rank if predicted, else 0) correlates with the rollout's F1 across
all 1000 rollouts. High correlation ⇒ predicting that contact early marks the good rollouts.

In [ ]:
ex = m.groupby('entry_id')['all_f1'].std().idxmax()
sub = cf[cf.entry_id == ex]
f1_by_r  = sub.drop_duplicates('r').set_index('r')['rollout_f1']
maxrank  = sub.groupby('r')['rank'].max()

recs = []
for (i, j), g in sub[sub.correct].groupby(['i','j']):
    early = pd.Series(0.0, index=f1_by_r.index)
    nr = g.groupby('r')['rank'].min()
    early.loc[nr.index] = 1 - (nr / maxrank.loc[nr.index]).values
    rho = np.corrcoef(early.values, f1_by_r.values)[0,1]
    recs.append((i, j, g['r'].nunique()/f1_by_r.shape[0], rho))
info = pd.DataFrame(recs, columns=['i','j','hit_rate','corr_earliness_f1'])
info = info.reindex(info.corr_earliness_f1.abs().sort_values(ascending=False).index)
print(f'target {ex}  (L={int(t[t.entry_id==ex].L.iloc[0])}, {f1_by_r.shape[0]} rollouts)')
display(info.head(10).round(3))

# left: F1 distribution for this target; right: earliness-vs-F1 for its top contact
fig, ax = plt.subplots(1, 2, figsize=(11,3.6))
ax[0].hist(f1_by_r.values, bins=40, color='#4C72B0'); ax[0].set_title(f'{ex}: F1 across rollouts'); ax[0].set_xlabel('F1')
top = info.iloc[0]
g = sub[(sub.i==top.i)&(sub.j==top.j)&(sub.correct)]
early = pd.Series(0.0, index=f1_by_r.index)
nr = g.groupby('r')['rank'].min(); early.loc[nr.index] = 1-(nr/maxrank.loc[nr.index]).values
binned = pd.DataFrame({'early':early.values,'f1':f1_by_r.values})
binned['bin'] = pd.cut(binned.early, [-.01,.001,.25,.5,.75,1.0], labels=['not pred','late','mid','early','earliest'])
binned.groupby('bin', observed=True)['f1'].mean().plot(kind='bar', ax=ax[1], color='#55A868')
ax[1].set_title(f'contact ({int(top.i)},{int(top.j)}): mean rollout F1 by how early it was predicted')
ax[1].set_ylabel('mean F1'); ax[1].set_xlabel('')
plt.tight_layout(); plt.show()

## Where to go from here

- These are aggregate scaffolds; the signal may be **per-target** — loop the section-2/3 probes over targets
  and look at the distribution, or condition on `L` / `n_gt`.
- `contacts_frame` has everything per contact (rank, sep, band, logprob, correct, residues) — slice it freely.
- Section 5 is the most promising thread: rank targets by how much a single informative contact explains their
  rollout variance, then look at what those contacts have in common.
- Join back to exp98's `rollout_metrics_all.parquet` on `entry_id`+`r` for the same rollouts at 5× more targets
  (order-independent features only — exp98's `pred` is sorted).

Data + method: [MarinFold#102](https://github.com/Open-Athena/MarinFold/issues/102),
experiment `experiments/exp102_rollout_accuracy_factors/` (see `analysis_starter.py` for the same analyses as a script).